In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# A：八次VAE编码与角度相关目标诊断

已填原四图diagnostic-20260909-152107-462244目录，无需改参数。单次运行自动完成：四图clean/+10各一次VAE编码，共8个观测缓存；随后CPU分析原优化解、收紧精度解、固定0.25°步长相关曲线及独立中心敏感性。0新生成、0内容评分，不需要CEG_WM_ROOT_KEY；仅需GPU runtime与已有HF_TOKEN访问同一public VAE。

缓存保存原reader实际接收的CHW float32观测：先在原VAE输出dtype完成shift/scale，再转float32，记录预处理配置与运算顺序。输出为Drive独立objective-cache-时间目录的8个NPZ、cache_rows.jsonl、rows.jsonl及report.json，保留失败单元。

正式reader不修改。真角只用于事后误差报告；0.25°离散曲线峰不等于连续全局最优，曲线与中心对照不参与正式变换选优。不能按当前图误差加固定角补偿，也不能由此宣称旋转已解决。前一轮真H已恢复内容；本轮定位优化器局部行为或相关峰本身偏移。

合成CPU检查已完成：原/tight最大角变化约0.0047°，不能解释真实图约1.12°偏差；近identity目标存在不光滑现象，tight不保证达到曲线最高点。真实VAE曲面需要此次观测，不再要求用户运行第二个入口。


In [ ]:
import json, os, pathlib, subprocess, sys, datetime
from importlib.metadata import version, PackageNotFoundError

REPO='https://github.com/RICHAAARC/CEG-WM.git'
BRANCH='dev/latent-sync-v1'
EXPECTED_EXACT='1a471f51af52183473b3c9ff1706cc457d030773'
checkout=pathlib.Path('/content/latent-sync-v1-github')
drive_root=pathlib.Path('/content/drive/MyDrive/CEG-WM/development/latent-sync-v1')
INPUT=drive_root/'diagnostic-20260909-152107-462244'
output_root=drive_root/('objective-cache-'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
if not checkout.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin',BRANCH],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','diffusers<0.40','transformers','accelerate','scipy','sentencepiece'],check=True)
environment={}
for package in ('torch','diffusers','numpy','scipy'):
    try: environment[package]=version(package)
    except PackageNotFoundError: environment[package]=None
print({'branch':BRANCH,'code':EXPECTED_EXACT,'versions':environment})
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
from google.colab import userdata
child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
child_env['OPENBLAS_NUM_THREADS']='1'  # avoid thread overhead in small CPU correlations
command=[sys.executable,'-m','experiments.run_latent_rotation_cache_diagnostic',
         '--mode','encode-and-diagnose','--input-dir',str(INPUT),'--output',str(output_root)]
completed=subprocess.run(command,cwd=checkout,env=child_env,check=False)
print('输出目录:', output_root)
if completed.returncode != 0:
    raise RuntimeError(f'诊断退出码 {completed.returncode}；已写入结果保留在 {output_root}')
report_path=output_root/'report.json'
if report_path.exists():
    report=json.loads(report_path.read_text())
    print({'report':str(report_path),'cached_observations':report.get('cached_observations'),'failed_units':report.get('failed_units')})
    for row in report.get('diagnostic_rows',[]):
        result=row.get('objective_diagnostic',{})
        print({'source':row['source'],'condition':row['condition'],'error':row['error'],
               'optimized_angles':{name:value['angle'] for name,value in result.get('optimizers',{}).items()},
               'sampled_curve_maximum':result.get('sampled_curve_maximum')})
